# Day 1 Demo — Notebook 2 of 3: RAG (Retrieval + Generation) — and where it breaks

Builds a retrieval + generation pipeline over the same 10-category dataset as
notebook 1, then demonstrates the core lesson of Day 1: fluent generation can
merge two unrelated retrieved records into one false, confident narrative —
even when retrieval itself found the right records individually.

## Setup (Colab)

Add `OPENAI_API_KEY` to Colab's Secrets manager (padlock icon, left sidebar) before running — no `.env` file needed on Colab.

In [1]:
%pip install -q sentence-transformers hnswlib openai python-dotenv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

from openai import OpenAI
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
print("Client ready")

Client ready


## Quick sanity check

In [4]:
response = client.chat.completions.create(
    model="gpt-4o-mini",  # cheap, fast, good enough for this project
    messages=[
        {"role": "user", "content": "In one sentence, what is Retrieval-Augmented Generation?"}
    ]
)

print(response.choices[0].message.content)

Retrieval-Augmented Generation (RAG) is a technique that combines generative models with information retrieval, allowing the model to enhance its responses by accessing and incorporating relevant external information from a specific dataset or knowledge base.


## Dataset — 10 varied categories (same corpus as notebook 1)

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

categories = {
    "Orders": [
        "Customer placed an order for laptops",
        "Client purchased new office chairs",
        "The account holder bought a batch of monitors",
        "Purchase order submitted for networking equipment",
        "New order received for printer cartridges",
        "Bulk order placed for server racks",
        "Customer requested a quote before ordering keyboards",
        "Order confirmation sent for the tablet shipment",
        "Client re-ordered the same set of webcams",
        "Purchase request approved for docking stations",
        "Customer added routers to their existing order",
        "Order placed for annual software licenses",
    ],
    "Billing": [
        "Invoice payment overdue by 45 days",
        "Billing statement shows a late payment fee",
        "Customer disputed a charge on their invoice",
        "Monthly billing cycle closed with no discrepancies",
        "Client requested a copy of their last invoice",
        "Payment reminder sent for outstanding balance",
        "Invoice was issued with an incorrect tax amount",
        "Customer set up autopay for future invoices",
        "Billing department flagged a duplicate charge",
        "Credit note issued against a cancelled invoice",
        "Client's payment bounced and needs reprocessing",
        "Invoice reconciliation completed for the quarter",
    ],
    "Shipping": [
        "Order shipped to customer's warehouse yesterday",
        "Product delivered to client's facility on time",
        "Shipment delayed due to a courier issue",
        "Package was returned to sender as undeliverable",
        "Tracking number issued for the outbound shipment",
        "Customer requested expedited delivery for their order",
        "Freight carrier confirmed pickup for the pallet",
        "Delivery attempted but no one was available",
        "Shipment split into two boxes due to weight limits",
        "Client's order arrived damaged in transit",
        "Logistics team rerouted the shipment through a new hub",
        "Customs held the international shipment for inspection",
    ],
    "Complaints": [
        "Customer support ticket opened for a defective laptop",
        "New complaint raised about a broken monitor",
        "Client reported the product stopped working after a week",
        "Support agent escalated the complaint to engineering",
        "Customer requested a replacement for the faulty unit",
        "Complaint closed after the refund was processed",
        "Client unhappy with the quality of the delivered item",
        "Support ticket reopened after the same issue recurred",
        "Customer threatened to cancel their contract over service quality",
        "Negative feedback submitted about installation support",
        "Client escalated their complaint to a account manager",
        "Repeated complaints logged about the same product line",
    ],
    "Contracts_Legal": [
        "Legal team reviewed the vendor contract renewal",
        "New service agreement signed with the client",
        "Contract clause disputed during the negotiation",
        "Non-disclosure agreement executed before the meeting",
        "Vendor contract terminated due to non-performance",
        "Legal flagged a liability clause in the draft agreement",
        "Master service agreement amended to include new terms",
        "Client requested an extension on the contract deadline",
        "Procurement finalized the terms of the supplier contract",
        "Contract renewal pending legal department sign-off",
        "Breach of contract notice sent to the vendor",
        "Service level agreement updated with new penalties",
    ],
    "HR_Payroll": [
        "Payroll processed for all employees this month",
        "New hire onboarding completed for the engineering team",
        "Employee submitted a request for annual leave",
        "HR flagged a discrepancy in the timesheet submission",
        "Performance review scheduled for next quarter",
        "Employee benefits enrollment window opened today",
        "Exit interview conducted for a departing team member",
        "Payroll correction issued for a missed overtime payment",
        "HR approved the internal transfer request",
        "New employee handbook distributed to all staff",
        "Recruitment team shortlisted candidates for the open role",
        "Employee raised a grievance about scheduling conflicts",
    ],
    "IT_Security": [
        "Security team detected unusual login activity",
        "IT support resolved a network outage in the office",
        "Password reset requested for a locked user account",
        "Firewall rules updated after the security audit",
        "Phishing attempt reported by an employee",
        "IT deployed a patch for the recent vulnerability",
        "Server downtime scheduled for maintenance overnight",
        "Access request approved for the new database",
        "Data backup completed successfully for all systems",
        "Security incident triggered an automatic account lockout",
        "IT ticket raised for a malfunctioning laptop",
        "VPN access restored after the configuration fix",
    ],
    "Sales_Leads": [
        "Sales rep followed up with a qualified lead",
        "New lead captured through the website contact form",
        "Prospect requested a product demo next week",
        "Sales pipeline updated with three new opportunities",
        "Lead converted into a paying customer",
        "Cold outreach campaign generated several new leads",
        "Sales team lost a deal to a competitor",
        "Discovery call scheduled with a potential enterprise client",
        "Lead marked as unqualified after initial screening",
        "Proposal sent to a prospect following their demo",
        "Sales forecast updated based on pipeline movement",
        "Referral lead came in from an existing customer",
    ],
    "Inventory": [
        "Warehouse stock count completed for the quarter",
        "Inventory levels low for a key product line",
        "Stock replenishment order triggered automatically",
        "Warehouse team flagged a discrepancy in stock records",
        "New shipment received and added to inventory",
        "Obsolete stock marked for clearance sale",
        "Inventory audit found several missing units",
        "Stock transferred between two regional warehouses",
        "Reorder point reached for the printer cartridge SKU",
        "Damaged inventory written off during the audit",
        "Warehouse capacity nearing its storage limit",
        "Cycle count scheduled for the electronics section",
    ],
    "Finance_Reports": [
        "Quarterly revenue report prepared for the board",
        "Finance team closed the books for the fiscal year",
        "Budget variance analysis completed for the department",
        "Expense report submitted for the recent business trip",
        "Finance flagged an unexpected increase in operating costs",
        "Annual financial statements sent for external audit",
        "Cash flow forecast updated for the next two quarters",
        "Cost center allocation reviewed by the finance team",
        "Profit margin analysis shared with senior leadership",
        "Finance approved the capital expenditure request",
        "Revenue recognition policy updated per new guidelines",
        "Departmental spending reconciled against the annual budget",
    ],
}

varied_texts = []
varied_labels = []
for category, sentences in categories.items():
    for s in sentences:
        varied_texts.append(s)
        varied_labels.append(category)

print(f"Total records: {len(varied_texts)}")
print(f"Categories: {len(categories)}")

varied_embeddings = model.encode(varied_texts, show_progress_bar=True)
print("Shape:", varied_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total records: 120
Categories: 10


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Shape: (120, 384)


## Rebuild the HNSW retrieval index

If HNSW is a social network of records, M decides how many friends each record gets to have, and ef_construction decides how carefully it searches for the right friends when the network is first being built. Neither of these affects how hard it searches later — that's ef's job at query time, which is the dial you already saw move the recall numbers

In [6]:
import numpy as np
import hnswlib

dim = varied_embeddings.shape[1]
varied_hnsw_index = hnswlib.Index(space='cosine', dim=dim)
varied_hnsw_index.init_index(max_elements=len(varied_texts), ef_construction=200, M=16)
varied_hnsw_index.add_items(
    np.ascontiguousarray(varied_embeddings.astype('float32')),
    np.arange(len(varied_texts))
)

## The RAG pipeline

Retrieve top-k similar records, hand them to the LLM as context, generate an answer.

In [7]:
def rag_query(question, k=3):
    q_emb = model.encode([question])[0]
    q_emb_f32 = np.ascontiguousarray(q_emb.astype('float32').reshape(1, -1))
    labels, distances = varied_hnsw_index.knn_query(q_emb_f32, k=k)
    retrieved_texts = [varied_texts[idx] for idx in labels[0]]

    context = "\n".join(f"- {t}" for t in retrieved_texts)
    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — don't use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "retrieved": retrieved_texts,
        "answer": response.choices[0].message.content
    }

# Straightforward, unambiguous question first
result = rag_query("Has a customer complained about a broken product recently?")
print("Retrieved context:")
for t in result["retrieved"]:
    print(f"  - {t}")
print("\nAnswer:", result["answer"])

Retrieved context:
  - Repeated complaints logged about the same product line
  - New complaint raised about a broken monitor
  - Customer requested a replacement for the faulty unit

Answer: Yes, a new complaint has been raised about a broken monitor.


## The break: ambiguous, overlapping-vocabulary questions

This is the live-demo moment — ask a question that straddles two categories that share vocabulary (Billing/Finance_Reports; Orders/Inventory) and watch generation merge them into one false narrative.

In [8]:
# Ambiguous between Billing and Finance_Reports (both concern money/reports)
result1 = rag_query("What financial issues came up this quarter?")

print("=== Query 1 ===")
print("Retrieved:")
for t in result1["retrieved"]:
    print(f"  - {t}")
print("Answer:", result1["answer"])

# Ambiguous between Orders and Inventory (both concern stock/replenishment)
result2 = rag_query("Tell me about the recent order for stock replenishment")

print("\n=== Query 2 ===")
print("Retrieved:")
for t in result2["retrieved"]:
    print(f"  - {t}")
print("Answer:", result2["answer"])

=== Query 1 ===
Retrieved:
  - Finance flagged an unexpected increase in operating costs
  - Cash flow forecast updated for the next two quarters
  - Finance team closed the books for the fiscal year
Answer: An unexpected increase in operating costs was flagged by Finance.

=== Query 2 ===
Retrieved:
  - Stock replenishment order triggered automatically
  - Order shipped to customer's warehouse yesterday
  - New shipment received and added to inventory
Answer: The recent order for stock replenishment was triggered automatically and was shipped to the customer's warehouse yesterday. A new shipment has been received and added to inventory.


## Checking against ground-truth categories

Each retrieved record's true category — shows retrieval actually found reasonable matches individually; it's the generation step that wrongly stitched them together.

In [9]:
def show_categories(retrieved_texts):
    for t in retrieved_texts:
        idx = varied_texts.index(t)  # safe here — this corpus has zero duplicates
        print(f"  [{varied_labels[idx]}] {t}")

print("Query 1 categories:")
show_categories(result1["retrieved"])
print("\nQuery 2 categories:")
show_categories(result2["retrieved"])

Query 1 categories:
  [Finance_Reports] Finance flagged an unexpected increase in operating costs
  [Finance_Reports] Cash flow forecast updated for the next two quarters
  [Finance_Reports] Finance team closed the books for the fiscal year

Query 2 categories:
  [Inventory] Stock replenishment order triggered automatically
  [Shipping] Order shipped to customer's warehouse yesterday
  [Inventory] New shipment received and added to inventory


## Does citation prompting fix it? (No — it's structural)

Explicitly instructing the model to cite sources and flag unrelated ones still produces a merged narrative. This confirms the issue isn't fixable by better prompting alone — it needs an explicit structure (notebook 3).

In [10]:
def rag_query_with_citations(question, k=3):
    q_emb = model.encode([question])[0]
    q_emb_f32 = np.ascontiguousarray(q_emb.astype('float32').reshape(1, -1))
    labels, distances = varied_hnsw_index.knn_query(q_emb_f32, k=k)
    retrieved_texts = [varied_texts[idx] for idx in labels[0]]

    context = "\n".join(f"[{i+1}] {t}" for i, t in enumerate(retrieved_texts))
    prompt = f"""Answer using ONLY the numbered context below. After each claim in your answer, cite the source number(s) it came from, like [1] or [1][2]. If two sources describe unrelated events, say so explicitly rather than combining them into one claim.

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(rag_query_with_citations("Tell me about the recent order for stock replenishment"))

The recent order for stock replenishment was triggered automatically [1]. Additionally, a new shipment related to this order was received and added to inventory [3].


---
Next: notebook 3 fixes this by backing the same question with an explicit graph instead of pure vector similarity.